# ema-second-moment — ex2: derive Adam's per-coordinate adaptive step-scale from the v buffer

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ema-second-moment`. Running the final beacon cell reports progress against the `Optimizer: Adam EMA second moment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA second moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-second-moment`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-second-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA second moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Adam adaptive step scale — `1 / (sqrt(v) + eps)`

Ex1 maintained the EMA second-moment buffer `v`. The deepening move is the next downstream step: derive the PER-COORDINATE step-scale that the Adam denominator produces.

```python
# After computing v_t = beta2*v + (1-beta2)*g^2:
step_scale = 1.0 / (v.sqrt() + eps)
# Coordinate-wise adaptive lr multiplier.
# Big |g| history → big v → SMALL step_scale.
# Small |g| history → small v → LARGE step_scale.
```

**Why eps is OUTSIDE the sqrt here (unlike BatchNorm).** Adam's eps primarily prevents divide-by-zero when v=0 (early steps before any grad accumulation). The PyTorch reference (and the original Kingma & Ba paper) uses `sqrt(v) + eps`. This is a deliberate departure from the BatchNorm convention — see the paper's appendix for the mean-square-error analysis.

**Operational meaning.** A coordinate with v=100 gets step_scale ≈ 0.1 — gradients are LARGE on this axis, so Adam takes SMALL steps. A coordinate with v=0.01 gets step_scale ≈ 10 — gradients are small on this axis, so Adam takes LARGE steps. That's adaptive per-parameter lr — the whole point of Adam over SGD.

**Watch out for v=0.** If v hasn't accumulated yet (or all observed g were 0), `step_scale = 1/eps` is huge. With eps=1e-8, that's a 1e8 lr multiplier — explosive. This is why Adam normally has a bias-correction step BEFORE the denominator AND why eps is usually set non-trivially (1e-7 or 1e-8, not 1e-30).

### Exercise 2 — derive Adam's per-coordinate adaptive step-scale from the v buffer

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Adam denominator formula `step_scale = 1 / (sqrt(v) + eps)` to a v-buffer tensor and verify the per-coordinate adaptive-lr signature — big-grad-history coords get small scales, small-grad-history coords get large scales.
> Keywords: adam, adaptive-lr, step-scale, second-moment
> ```

**KCs targeted:** `step-scale-equals-one-over-sqrt-v-plus-eps`, `big-v-small-scale-small-v-big-scale`

Implement `ex2_adam_step_scale(v, eps)`. The deepening variant of ex1.

Inputs:
- `v`: `torch.Tensor`, the Adam second-moment buffer (elementwise, non-negative). May contain zeros (early-step buffer).
- `eps`: `float`, the Adam epsilon (typical 1e-8).

Compute:

`step_scale = 1.0 / (v.sqrt() + eps)`

Return a dict with EXACTLY these keys:

- `'step_scale'`: `torch.Tensor`, same shape as `v`.
- `'step_scale_at_v_zero'`: `float`, `1.0 / eps` — the maximum achievable step-scale (when v=0).
- `'max_step_scale'`: `float`, `step_scale.max().item()`. Must be `<= step_scale_at_v_zero + 1e-6`.
- `'min_step_scale'`: `float`, `step_scale.min().item()`.
- `'v_at_max_scale_idx'`: `int`, the FLAT index of the v entry that produced the LARGEST step_scale (use `v.argmin().item()` — smallest v → largest scale).
- `'v_at_min_scale_idx'`: `int`, `v.argmax().item()` — largest v → smallest scale.

Constraints:
- Use `v.sqrt()` not `t.sqrt(v)` (equivalent, but consistent with ARENA's tensor-method style).
- Use Python `/` and `+` — they dispatch to elementwise ops on tensors.
- Do not mutate `v`.

In [ ]:
def ex2_adam_step_scale(v, eps):
    step_scale = 1.0 / (v.sqrt() + eps)
    return {
        'step_scale': step_scale,
        'step_scale_at_v_zero': 1.0 / eps,
        'max_step_scale': step_scale.max().item(),
        'min_step_scale': step_scale.min().item(),
        'v_at_max_scale_idx': v.argmin().item(),
        'v_at_min_scale_idx': v.argmax().item(),
    }


<details><summary>Solution</summary>

```python
def ex2_adam_step_scale(v, eps):
    step_scale = 1.0 / (v.sqrt() + eps)
    return {
        'step_scale': step_scale,
        'step_scale_at_v_zero': 1.0 / eps,
        'max_step_scale': step_scale.max().item(),
        'min_step_scale': step_scale.min().item(),
        'v_at_max_scale_idx': v.argmin().item(),
        'v_at_min_scale_idx': v.argmax().item(),
    }
```

**`v.argmin()` finds the MAX-scale coordinate.** Because `step_scale` is strictly decreasing in `v` (when v >= 0), the smallest v gives the largest scale. We can find the argmax of `step_scale` directly OR the argmin of `v` — same index. Doing it via `v.argmin()` avoids needing the step_scale tensor for the lookup.

**Why eps is OUTSIDE the sqrt for Adam.** Unlike BatchNorm (where eps INSIDE the sqrt bounds the backward gradient), Adam's eps is a forward-pass division floor. The Kingma & Ba paper has it outside; PyTorch follows. With eps=1e-8 and v=0, step_scale = 1e8 — huge but finite.

**The 1e8 step-scale ceiling is the early-training trap.** Before grad accumulation, v is near zero everywhere. Adam's denominator is tiny → step_scale is huge → updates can be explosive. The standard fix is BIAS CORRECTION (divide v by `1 - beta2^t`, which is small at t=1 so v_hat is INFLATED to the steady-state magnitude). That correction is a separate atom — this drill focuses just on the scale formula.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()